# SQLite-VSS

>[SQLite-VSS](https://alexgarcia.xyz/sqlite-vss/) 是一个为向量搜索设计的 `SQLite` 扩展，它强调本地优先操作，并且易于集成到应用程序中，无需外部服务器。它利用 `Faiss` 库，提供了高效的相似性搜索和聚类功能。

您需要安装 `langchain-community` 才能使用此集成，命令为 `pip install -qU langchain-community`

本笔记本展示了如何使用 `SQLiteVSS` 向量数据库。

In [ ]:
# You need to install sqlite-vss as a dependency.
%pip install --upgrade --quiet  sqlite-vss

## 快速入门

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from langchain_community.vectorstores import SQLiteVSS
from langchain_text_splitters import CharacterTextSplitter

# load the document and split it into chunks
loader = TextLoader("../../how_to/state_of_the_union.txt")
documents = loader.load()

# split it into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)
texts = [doc.page_content for doc in docs]


# create the open-source embedding function
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


# load it in sqlite-vss in a table named state_union.
# the db_file parameter is the name of the file you want
# as your sqlite database.
db = SQLiteVSS.from_texts(
    texts=texts,
    embedding=embedding_function,
    table="state_union",
    db_file="/tmp/vss.db",
)

# query it
query = "What did the president say about Ketanji Brown Jackson"
data = db.similarity_search(query)

# print results
data[0].page_content

'Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'

## 使用现有的 SQLite 连接

You can pass an existing SQLite connection to the `create_engine` function.

您可以将现有的 SQLite 连接传递给 `create_engine` 函数。

```python
from sqlalchemy import create_engine
import sqlite3

# Create a SQLite connection object
conn = sqlite3.connect("my_database.db")

# Pass the connection to create_engine
engine = create_engine("sqlite:///:memory:", connect_args={"creator": lambda: conn})

# Now you can use the engine to interact with the database
# For example, to execute a query:
with engine.connect() as connection:
    result = connection.execute("SELECT 'Hello, world!'")
    for row in result:
        print(row[0])
```

In this example, we first create a standard `sqlite3.Connection` object. Then, we pass this connection to `create_engine` using the `connect_args` parameter. The `creator` argument in `connect_args` is a callable that returns a DB-API 2 connection object. In this case, we provide a lambda function that simply returns our existing `conn` object.

在此示例中，我们首先创建一个标准的 `sqlite3.Connection` 对象。然后，我们使用 `connect_args` 参数将此连接传递给 `create_engine`。`connect_args` 中的 `creator` 参数是一个可调用对象，它返回一个 DB-API 2 连接对象。在这种情况下，我们提供了一个 lambda 函数，该函数仅返回我们现有的 `conn` 对象。

This approach is useful when you already have an active SQLite connection and want to leverage SQLAlchemy's features, such as its ORM or connection pooling, with that existing connection.

这种方法在您已经拥有活动的 SQLite 连接并希望利用 SQLAlchemy 的功能（例如其 ORM 或连接池）与该现有连接进行交互时非常有用。

In [7]:
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from langchain_community.vectorstores import SQLiteVSS
from langchain_text_splitters import CharacterTextSplitter

# load the document and split it into chunks
loader = TextLoader("../../how_to/state_of_the_union.txt")
documents = loader.load()

# split it into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)
texts = [doc.page_content for doc in docs]


# create the open-source embedding function
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
connection = SQLiteVSS.create_connection(db_file="/tmp/vss.db")

db1 = SQLiteVSS(
    table="state_union", embedding=embedding_function, connection=connection
)

db1.add_texts(["Ketanji Brown Jackson is awesome"])
# query it again
query = "What did the president say about Ketanji Brown Jackson"
data = db1.similarity_search(query)

# print results
data[0].page_content

'Ketanji Brown Jackson is awesome'

In [13]:
# Cleaning up
import os

os.remove("/tmp/vss.db")